In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
    cross_val_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

from xgboost import XGBRegressor
import joblib

# 1. Dataset Load
df = pd.read_csv("../data/processed/featured_superstore.csv")

# Standardize column names just in case
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')

# ------------------------------------------------------------------
# 🔥 STEP 1: OUTLIER HANDLING (Top 1% Extreme Sales Cap karna)
# ------------------------------------------------------------------
# 99th percentile se upar ki extreme entries ko filter kar rahe hain 
# taaki model normal transactions par master kar sake
upper_limit = df['sales'].quantile(0.99)
df_model = df[df['sales'] <= upper_limit].copy()

# ------------------------------------------------------------------
# 🔥 STEP 2: HIGH-IMPACT FEATURE ENGINEERING (Target Encoding)
# ------------------------------------------------------------------
# Har Sub-Category ka median sales calculate karke naya feature banana
subcat_median_sales = df_model.groupby('sub_category')['sales'].transform('median')
df_model['subcat_expected_price'] = subcat_median_sales

# ------------------------------------------------------------------
# 🔥 STEP 3: FEATURE SELECTION
# ------------------------------------------------------------------
y = df_model['sales']

# Direct high-signal features list
feature_cols = [
    'quantity', 'discount', 'subcat_expected_price', 
    'order_month', 'order_year', 'order_quarter', 'is_weekend',
    'category', 'sub_category', 'market', 'segment', 'ship_mode'
]

# Agar dataset me 'shipping_cost' aur 'profit' hai toh unhe add kar lo (Massive Boost)
if 'shipping_cost' in df_model.columns:
    feature_cols.append('shipping_cost')
if 'profit' in df_model.columns:
    feature_cols.append('profit')

X = df_model[feature_cols]

# Separate Numeric and Categorical
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

# ------------------------------------------------------------------
# 🔥 STEP 4: TRAIN-TEST SPLIT & MODEL PIPELINE
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# =====================================================
# XGBOOST + GRID SEARCH CV
# =====================================================

xgb_model = XGBRegressor(
    random_state=42,
    n_jobs=-1
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", xgb_model)
])

param_grid = {

    "regressor__n_estimators": [300, 500, 700],

    "regressor__learning_rate": [0.03, 0.05, 0.1],

    "regressor__max_depth": [5, 7, 9],

    "regressor__subsample": [0.8, 1.0],

    "regressor__colsample_bytree": [0.8, 1.0]

}

grid_search = GridSearchCV(

    estimator=xgb_pipeline,

    param_grid=param_grid,

    cv=5,

    scoring="r2",

    n_jobs=-1,

    verbose=2

)

print("Running GridSearchCV...")

grid_search.fit(X_train, y_train)

print("Grid Search Completed!")

print("\nBest Parameters:")
print(grid_search.best_params_)

print(f"Best Cross Validation R² : {grid_search.best_score_:.4f}")

xgb_pipeline = grid_search.best_estimator_

# ==========================================
# SAVE MODEL PREDICTIONS
# ==========================================
y_pred = xgb_pipeline.predict(X_test)

prediction_df = pd.DataFrame({

    "Actual Sales": y_test.values,

    "Predicted Sales": y_pred

})

joblib.dump(

    prediction_df,

    "../models/prediction_results.pkl"

)

print("Prediction Results Saved!")

# =====================================================
# 5-FOLD CROSS VALIDATION
# =====================================================

print("\nPerforming 5-Fold Cross Validation...")

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("\n==========================================")
print("5-Fold Cross Validation Results")
print("==========================================")

for i, score in enumerate(cv_scores, start=1):
    print(f"Fold {i} R² Score : {score:.4f}")

print("------------------------------------------")
print(f"Average R² Score : {cv_scores.mean():.4f}")
print(f"Standard Deviation : {cv_scores.std():.4f}")
print("==========================================")

# ------------------------------------------------------------------
# 🔥 STEP 5: EVALUATION
# ------------------------------------------------------------------

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n==========================================")
print(f"🎯 FINAL MODEL R² SCORE : {r2:.4f}")
print(f"💵 MAE ($)              : ${mae:.2f}")
print(f"📉 RMSE ($)             : ${rmse:.2f}")
print("==========================================")

# Save the High Accuracy Model
joblib.dump(xgb_pipeline, "../models/xgboost_sales_model.pkl")
print("Model saved to '../models/xgboost_sales_model.pkl'")

Running GridSearchCV...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Grid Search Completed!

Best Parameters:
{'regressor__colsample_bytree': 0.8, 'regressor__learning_rate': 0.03, 'regressor__max_depth': 7, 'regressor__n_estimators': 500, 'regressor__subsample': 1.0}
Best Cross Validation R² : 0.8894
Prediction Results Saved!

Performing 5-Fold Cross Validation...

5-Fold Cross Validation Results
Fold 1 R² Score : 0.8846
Fold 2 R² Score : 0.8933
Fold 3 R² Score : 0.8892
Fold 4 R² Score : 0.8915
Fold 5 R² Score : 0.8955
------------------------------------------
Average R² Score : 0.8908
Standard Deviation : 0.0037

🎯 FINAL MODEL R² SCORE : 0.8853
💵 MAE ($)              : $52.11
📉 RMSE ($)             : $112.74
Model saved to '../models/xgboost_sales_model.pkl'


In [12]:
# ==========================================
# SAVE MODEL MATRIX
# ==========================================

# Preprocessor ko fit karke transformed features nikalo
X_processed = xgb_pipeline.named_steps["preprocessor"].transform(X)

# Feature Names
feature_names = xgb_pipeline.named_steps["preprocessor"].get_feature_names_out()

# Dictionary Save Karna
model_matrix = {
    "X_processed": X_processed,
    "feature_names": feature_names,
    "numeric_columns": num_cols,
    "categorical_columns": cat_cols,
    "training_columns": feature_cols
}

joblib.dump(model_matrix, "../models/model_matrix.pkl")

print("Model Matrix Saved Successfully!")

Model Matrix Saved Successfully!


In [13]:
# ==========================================
# FEATURE IMPORTANCE
# ==========================================

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_pipeline.named_steps["regressor"].feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

joblib.dump(feature_importance, "../models/feature_importance.pkl")

print("Feature Importance Saved!")

Feature Importance Saved!


In [15]:
# ==========================================
# SAVE MODEL METRICS
# ==========================================
metrics = {

    "R2 Score": cv_scores.mean(),

    "MAE": mae,

    "RMSE": rmse,

    "CV Mean R2": cv_scores.mean(),

    "CV Std": cv_scores.std()

}

joblib.dump(
    metrics,
    "../models/model_metrics.pkl"
)

print("Model Metrics Saved Successfully!")

Model Metrics Saved Successfully!
